In [ ]:
import os
from dotenv import load_dotenv

load_dotenv()

def get_groq_key() -> str:
    # Streamlit Cloud secrets take priority
    try:
        import streamlit as st
        key = st.secrets.get("GROQ_API_KEY", "")
        if key:
            return key
    except Exception:
        pass
    # Fall back to .env / environment variable
    return os.getenv("GROQ_API_KEY", "")

# Groq API
GROQ_MODEL = "llama-3.3-70b-versatile"

# Embeddings — runs locally, no API key needed
EMBEDDING_MODEL = "all-MiniLM-L6-v2"

# Chunking
CHUNK_SIZE = 1000
CHUNK_OVERLAP = 200

# Chat memory
MAX_CONTEXT_TURNS = 6
RETRIEVER_K = 3

# Paths
DOCS_DIR = "sample_docs"
VECTORSTORE_PATH = "vectorstore_index"

In [ ]:
import os
import tempfile
import streamlit as st
from pathlib import Path

from document_loader import load_directory, load_uploaded_file
from rag_pipeline import RAGChatbot
from config import DOCS_DIR, get_groq_key

st.set_page_config(
    page_title="RAG Chatbot",
    page_icon="🤖",
    layout="centered",
    initial_sidebar_state="collapsed",
)

st.markdown("""
<style>
/* ── Base ── */
.stApp { background-color: #f4f6fb; }
[data-testid="collapsedControl"] { display: none; }
[data-testid="stSidebar"] { display: none; }

/* ── Nav bar ── */
.nav-bar {
    display: flex;
    align-items: center;
    justify-content: space-between;
    padding: 14px 24px;
    background: #ffffff;
    border-bottom: 1px solid #e3e6f0;
    border-radius: 0 0 14px 14px;
    box-shadow: 0 2px 10px rgba(0,0,0,0.06);
    margin-bottom: 32px;
}
.nav-logo { font-size: 20px; font-weight: 800; color: #1e2245; }
.nav-logo span { color: #4f6ef7; }
.nav-badge {
    background: #eef1ff;
    color: #4f6ef7;
    font-size: 12px;
    font-weight: 600;
    padding: 4px 14px;
    border-radius: 20px;
    border: 1px solid #d4dcff;
}

/* ── Page header ── */
.page-header { text-align: center; margin: 8px 0 32px; }
.page-header h1 { font-size: 30px; font-weight: 800; color: #1e2245; margin-bottom: 8px; }
.page-header p  { font-size: 15px; color: #6b7280; }

/* ── Upload cards ── */
.upload-card {
    background: #ffffff;
    border: 1px solid #e3e6f0;
    border-radius: 14px;
    padding: 22px 26px 14px;
    margin: 14px 0;
    box-shadow: 0 2px 8px rgba(0,0,0,0.04);
}
.card-title { font-size: 15px; font-weight: 700; color: #1e2245; margin-bottom: 4px; }
.card-sub   { font-size: 13px; color: #6b7280; margin-bottom: 14px; }

/* ── Status pill ── */
.status-pill {
    display: inline-flex;
    align-items: center;
    gap: 6px;
    background: #f0fdf4;
    color: #16a34a;
    border: 1px solid #bbf7d0;
    border-radius: 20px;
    padding: 4px 14px;
    font-size: 12px;
    font-weight: 600;
}
.status-pill-warn {
    display: inline-flex;
    align-items: center;
    gap: 6px;
    background: #fffbeb;
    color: #b45309;
    border: 1px solid #fde68a;
    border-radius: 20px;
    padding: 4px 14px;
    font-size: 12px;
    font-weight: 600;
}

/* ── Chat page header ── */
.chat-topbar {
    background: #ffffff;
    border-bottom: 1px solid #e3e6f0;
    border-radius: 14px 14px 0 0;
    padding: 14px 20px;
    margin-bottom: 0;
    box-shadow: 0 2px 6px rgba(0,0,0,0.04);
}

/* ── Message bubbles ── */
.user-row {
    display: flex;
    justify-content: flex-end;
    align-items: flex-end;
    gap: 8px;
    margin: 10px 0;
}
.user-bubble {
    background: #4f6ef7;
    color: #ffffff;
    padding: 11px 16px;
    border-radius: 18px 18px 4px 18px;
    max-width: 72%;
    font-size: 14px;
    line-height: 1.55;
    word-wrap: break-word;
}
.user-avatar {
    width: 32px; height: 32px;
    border-radius: 50%;
    background: #dde3ff;
    display: flex; align-items: center; justify-content: center;
    font-size: 15px; flex-shrink: 0;
}

.bot-row {
    display: flex;
    justify-content: flex-start;
    align-items: flex-end;
    gap: 8px;
    margin: 10px 0;
}
.bot-avatar {
    width: 32px; height: 32px;
    border-radius: 50%;
    background: #f0f4ff;
    border: 1px solid #d4dcff;
    display: flex; align-items: center; justify-content: center;
    font-size: 15px; flex-shrink: 0;
}
.bot-bubble {
    background: #ffffff;
    color: #1e2245;
    padding: 11px 16px;
    border-radius: 4px 18px 18px 18px;
    max-width: 72%;
    font-size: 14px;
    line-height: 1.6;
    border: 1px solid #e3e6f0;
    box-shadow: 0 1px 4px rgba(0,0,0,0.05);
    word-wrap: break-word;
}
.source-tag {
    font-size: 11px; color: #9ca3af;
    margin: 2px 0 10px 40px;
}

/* ── Empty chat hint ── */
.hint-box { text-align: center; padding: 50px 20px; }
.hint-icon  { font-size: 44px; margin-bottom: 10px; }
.hint-title { font-size: 17px; font-weight: 600; color: #374151; margin-bottom: 6px; }
.hint-sub   { font-size: 13px; color: #9ca3af; }
</style>
""", unsafe_allow_html=True)


# ── Session state ──────────────────────────────────────────────────────────

def _init():
    defaults = {
        "chatbot": None,
        "messages": [],
        "docs_loaded": False,
        "page": "upload",
        "doc_count": 0,
        "ds_key": get_groq_key(),
    }
    for k, v in defaults.items():
        if k not in st.session_state:
            st.session_state[k] = v

_init()


def _handle_error(e: Exception):
    err = str(e)
    if "1455" in err or "paging file" in err.lower():
        st.error(
            "Not enough virtual memory (Windows error 1455). "
            "Go to **System Properties → Advanced → Performance → Virtual Memory** "
            "and increase your paging file size, then restart."
        )
    else:
        st.error(f"Error: {e}")


# ── UPLOAD PAGE ────────────────────────────────────────────────────────────

def render_upload_page():
    st.markdown("""
    <div class="nav-bar">
        <div class="nav-logo">🤖 RAG<span>Chat</span></div>
        <div class="nav-badge">Free · Local · No API Key</div>
    </div>
    """, unsafe_allow_html=True)

    st.markdown("""
    <div class="page-header">
        <h1>Build Your Knowledge Base</h1>
        <p>Load documents to chat with — use the built-in samples or upload your own files.</p>
    </div>
    """, unsafe_allow_html=True)

    # ── Sample docs ────────────────────────────────────────────────────
    st.markdown("""
    <div class="upload-card">
        <div class="card-title">📚 Step 1 — Load Sample Documents</div>
        <div class="card-sub">Built-in knowledge base covering AI, Machine Learning, and Python programming.</div>
    </div>
    """, unsafe_allow_html=True)

    if st.button("⚡ Load Sample Docs", use_container_width=True, type="primary"):
        with st.spinner("Embedding documents — this may take a moment on first run…"):
            try:
                docs = load_directory(DOCS_DIR)
                if not docs:
                    st.error("No documents found in sample_docs/")
                else:
                    if st.session_state.chatbot is None:
                        st.session_state.chatbot = RAGChatbot(api_key=st.session_state.ds_key)
                    st.session_state.chatbot.build_vectorstore(docs)
                    st.session_state.docs_loaded = True
                    st.session_state.doc_count = len(docs)
                    st.success(f"✅ {len(docs)} chunks loaded! Click **Start Chatting** below.")
            except Exception as e:
                _handle_error(e)

    st.markdown("<div style='margin:20px 0'></div>", unsafe_allow_html=True)

    # ── Upload files ───────────────────────────────────────────────────
    st.markdown("""
    <div class="upload-card">
        <div class="card-title">📄 Step 2 — Upload Your Own Files (Optional)</div>
        <div class="card-sub">Supports PDF and TXT files. Upload one or multiple files at once.</div>
    </div>
    """, unsafe_allow_html=True)

    uploaded_files = st.file_uploader(
        "Drop your PDF or TXT files here",
        type=["txt", "pdf"],
        accept_multiple_files=True,
    )

    if uploaded_files:
        for f in uploaded_files:
            icon = "📕" if Path(f.name).suffix.upper() == ".PDF" else "📄"
            st.markdown(f"{icon} **{f.name}**")

        if st.button("📥 Process & Ingest Files", use_container_width=True):
            all_docs = []
            with st.spinner("Processing files…"):
                try:
                    for uf in uploaded_files:
                        suffix = Path(uf.name).suffix
                        with tempfile.NamedTemporaryFile(delete=False, suffix=suffix) as tmp:
                            tmp.write(uf.read())
                            tmp_path = tmp.name
                        all_docs.extend(load_uploaded_file(tmp_path))
                        os.unlink(tmp_path)

                    if not all_docs:
                        st.error("Could not extract text from the uploaded files.")
                    else:
                        if st.session_state.chatbot is None:
                            st.session_state.chatbot = RAGChatbot(api_key=st.session_state.ds_key)
                        st.session_state.chatbot.add_documents(all_docs)
                        st.session_state.docs_loaded = True
                        st.session_state.doc_count += len(all_docs)
                        st.success(f"✅ {len(all_docs)} chunks ingested! Click **Start Chatting** below.")
                except Exception as e:
                    _handle_error(e)

    st.markdown("<div style='margin:24px 0'></div>", unsafe_allow_html=True)

    # ── Start chatting ─────────────────────────────────────────────────
    if st.session_state.docs_loaded:
        st.markdown(f"""
        <div style="text-align:center;margin:8px 0 16px">
            <span class="status-pill">✅ Knowledge base ready — {st.session_state.doc_count} chunks loaded</span>
        </div>
        """, unsafe_allow_html=True)
        if st.button("💬 Start Chatting →", use_container_width=True, type="primary"):
            st.session_state.page = "chat"
            st.rerun()
    else:
        st.markdown("""
        <div style="text-align:center;margin:8px 0 16px">
            <span class="status-pill-warn">⚠️ No documents loaded yet</span>
        </div>
        """, unsafe_allow_html=True)


# ── CHAT PAGE ──────────────────────────────────────────────────────────────

def render_chat_page():
    # Top bar
    col_back, col_title, col_status = st.columns([1, 5, 2])
    with col_back:
        if st.button("← Back", help="Return to upload page"):
            st.session_state.page = "upload"
            st.rerun()
    with col_title:
        st.markdown(
            "<h3 style='margin:0;padding:5px 0;color:#1e2245;font-size:20px'>🤖 RAG Chatbot</h3>",
            unsafe_allow_html=True,
        )
    with col_status:
        st.markdown(
            f"<div style='text-align:right;padding:6px 0'>"
            f"<span class='status-pill'>✅ {st.session_state.doc_count} chunks</span></div>",
            unsafe_allow_html=True,
        )

    st.markdown("<hr style='border:none;border-top:1px solid #e3e6f0;margin:6px 0 20px'>", unsafe_allow_html=True)

    # Messages
    if not st.session_state.messages:
        st.markdown("""
        <div class="hint-box">
            <div class="hint-icon">💬</div>
            <div class="hint-title">Ask anything about your documents</div>
            <div class="hint-sub">Your knowledge base is ready. Type a question below to get started.</div>
        </div>
        """, unsafe_allow_html=True)
        col_a, col_b, col_c = st.columns(3)
        with col_a:
            st.info("What is machine learning?")
        with col_b:
            st.info("How does RAG work?")
        with col_c:
            st.info("What is deep learning?")
    else:
        for msg in st.session_state.messages:
            if msg["role"] == "user":
                st.markdown(f"""
                <div class="user-row">
                    <div class="user-bubble">{msg["content"]}</div>
                    <div class="user-avatar">👤</div>
                </div>""", unsafe_allow_html=True)
            else:
                st.markdown(f"""
                <div class="bot-row">
                    <div class="bot-avatar">🤖</div>
                    <div class="bot-bubble">{msg["content"]}</div>
                </div>""", unsafe_allow_html=True)
                if msg.get("sources"):
                    unique = list({s.metadata.get("source", "") for s in msg["sources"]})
                    names = " · ".join(f"📎 {Path(s).name}" for s in unique if s)
                    if names:
                        st.markdown(f'<div class="source-tag">{names}</div>', unsafe_allow_html=True)

    st.markdown("<br>", unsafe_allow_html=True)

    # Input form
    with st.form(key="chat_form", clear_on_submit=True):
        col_input, col_btn = st.columns([6, 1])
        with col_input:
            user_input = st.text_input(
                "message",
                placeholder="Ask a question about your documents…",
                label_visibility="collapsed",
            )
        with col_btn:
            submitted = st.form_submit_button("Send ➤", use_container_width=True, type="primary")

    col_clear, _ = st.columns([1, 5])
    with col_clear:
        if st.button("🗑️ Clear Chat", use_container_width=True):
            st.session_state.messages = []
            if st.session_state.chatbot:
                st.session_state.chatbot.clear_memory()
            st.rerun()

    if submitted and user_input.strip():
        question = user_input.strip()
        st.session_state.messages.append({"role": "user", "content": question})

        # Show user bubble immediately
        st.markdown(f"""
        <div class="user-row">
            <div class="user-bubble">{question}</div>
            <div class="user-avatar">👤</div>
        </div>""", unsafe_allow_html=True)

        # Stream bot response into a live placeholder
        placeholder = st.empty()
        full_response = ""
        sources = []

        try:
            for token in st.session_state.chatbot.chat_stream(question):
                full_response += token
                placeholder.markdown(f"""
                <div class="bot-row">
                    <div class="bot-avatar">🤖</div>
                    <div class="bot-bubble">{full_response}▌</div>
                </div>""", unsafe_allow_html=True)

            # Final render — remove blinking cursor
            placeholder.markdown(f"""
            <div class="bot-row">
                <div class="bot-avatar">🤖</div>
                <div class="bot-bubble">{full_response}</div>
            </div>""", unsafe_allow_html=True)

            sources = getattr(st.session_state.chatbot, "last_sources", [])

        except Exception as e:
            err = str(e)
            if "1455" in err or "paging file" in err.lower():
                full_response = (
                    "⚠️ Windows ran out of virtual memory (error 1455). "
                    "Increase your Page File size via System Properties → "
                    "Advanced → Performance → Virtual Memory, then restart."
                )
            else:
                full_response = (
                    f"⚠️ Error: {e}\n\n"
                    "Check your Groq API key at console.groq.com"
                )
            placeholder.markdown(f"""
            <div class="bot-row">
                <div class="bot-avatar">🤖</div>
                <div class="bot-bubble">{full_response}</div>
            </div>""", unsafe_allow_html=True)

        st.session_state.messages.append(
            {"role": "assistant", "content": full_response, "sources": sources}
        )
        st.rerun()


# ── Router ─────────────────────────────────────────────────────────────────

if st.session_state.page == "upload":
    render_upload_page()
else:
    render_chat_page()

In [ ]:
import os
from pathlib import Path
from langchain_core.documents import Document
from langchain_community.document_loaders import TextLoader, PyPDFLoader
from config import CHUNK_SIZE, CHUNK_OVERLAP


def _split_text(text: str, source: str = "") -> list:
    """Simple recursive text splitter — no langchain_text_splitters needed."""
    separators = ["\n\n", "\n", ". ", " ", ""]
    chunks = []

    def _recurse(fragment: str, seps: list):
        if len(fragment) <= CHUNK_SIZE:
            if fragment.strip():
                chunks.append(fragment.strip())
            return
        sep = seps[0] if seps else ""
        parts = fragment.split(sep) if sep else [fragment[i:i+1] for i in range(len(fragment))]
        current = ""
        for part in parts:
            candidate = (current + sep + part) if current else part
            if len(candidate) <= CHUNK_SIZE:
                current = candidate
            else:
                if current.strip():
                    chunks.append(current.strip())
                if len(part) > CHUNK_SIZE and len(seps) > 1:
                    _recurse(part, seps[1:])
                else:
                    current = part
        if current.strip():
            chunks.append(current.strip())

    _recurse(text, separators)

    # Apply overlap: each chunk includes tail of previous chunk
    docs = []
    for i, chunk in enumerate(chunks):
        if i > 0 and CHUNK_OVERLAP > 0:
            prev_tail = chunks[i - 1][-CHUNK_OVERLAP:]
            chunk = prev_tail + " " + chunk
        docs.append(Document(page_content=chunk, metadata={"source": source}))
    return docs


def load_directory(source_dir: str) -> list:
    docs = []
    for file_path in Path(source_dir).rglob("*"):
        try:
            if file_path.suffix.lower() == ".pdf":
                loader = PyPDFLoader(str(file_path))
                raw = loader.load()
                for page in raw:
                    docs.extend(_split_text(page.page_content, source=str(file_path)))
            elif file_path.suffix.lower() in (".txt", ".md"):
                loader = TextLoader(str(file_path), encoding="utf-8")
                raw = loader.load()
                for doc in raw:
                    docs.extend(_split_text(doc.page_content, source=str(file_path)))
        except Exception:
            continue
    return docs


def load_uploaded_file(tmp_path: str) -> list:
    ext = Path(tmp_path).suffix.lower()
    if ext == ".pdf":
        loader = PyPDFLoader(tmp_path)
    else:
        loader = TextLoader(tmp_path, encoding="utf-8")
    raw = loader.load()
    docs = []
    for doc in raw:
        docs.extend(_split_text(doc.page_content, source=tmp_path))
    return docs

In [ ]:
from __future__ import annotations

from langchain_groq import ChatGroq
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_classic.memory import ConversationBufferWindowMemory
from langchain_core.prompts import PromptTemplate

from config import (
    get_groq_key, GROQ_MODEL,
    EMBEDDING_MODEL, MAX_CONTEXT_TURNS, RETRIEVER_K,
)

_QA_TEMPLATE = """You are a helpful AI assistant with access to a knowledge base.
Use the retrieved context below to answer the question as accurately as possible.

Guidelines:
- If the answer is in the context, answer directly.
- If the context is only partly helpful, combine it with your general knowledge and say so.
- If the context is irrelevant, answer from general knowledge and note the knowledge base did not cover this topic.
- Be concise, clear, and friendly.

Context:
{context}

Chat History:
{chat_history}

Question: {question}

Answer:"""

_CONDENSE_TEMPLATE = """Given the conversation history and a follow-up question, \
rephrase the follow-up as a standalone question that captures the full intent.

Chat History:
{chat_history}

Follow-Up: {question}

Standalone Question:"""

# Singleton — load embedding model once per process
_embeddings_singleton: HuggingFaceEmbeddings | None = None


def _get_embeddings() -> HuggingFaceEmbeddings:
    global _embeddings_singleton
    if _embeddings_singleton is None:
        _embeddings_singleton = HuggingFaceEmbeddings(
            model_name=EMBEDDING_MODEL,
            model_kwargs={"device": "cpu"},
            encode_kwargs={"normalize_embeddings": True},
        )
    return _embeddings_singleton


class RAGChatbot:
    def __init__(self, api_key: str = ""):
        self.api_key = api_key or get_groq_key()
        self.embeddings = _get_embeddings()
        self.vectorstore: FAISS | None = None
        self.last_sources: list = []
        self._reset_memory()

    def _reset_memory(self):
        self.memory = ConversationBufferWindowMemory(
            k=MAX_CONTEXT_TURNS,
            memory_key="chat_history",
            return_messages=False,
            output_key="answer",
        )

    def clear_memory(self):
        self._reset_memory()

    def build_vectorstore(self, documents: list) -> "RAGChatbot":
        self.vectorstore = FAISS.from_documents(documents, self.embeddings)
        return self

    def add_documents(self, documents: list) -> "RAGChatbot":
        if self.vectorstore is None:
            return self.build_vectorstore(documents)
        self.vectorstore.add_documents(documents)
        return self

    @property
    def has_documents(self) -> bool:
        return self.vectorstore is not None

    def _get_llm(self, streaming: bool = False) -> ChatGroq:
        if not self.api_key:
            raise ValueError("Groq API key is required. Enter it in Step 1.")
        return ChatGroq(
            model=GROQ_MODEL,
            groq_api_key=self.api_key,
            temperature=0.7,
            max_tokens=1024,
            streaming=streaming,
        )

    def _format_history(self) -> str:
        messages = self.memory.chat_memory.messages
        if not messages:
            return ""
        pairs = []
        for i in range(0, len(messages) - 1, 2):
            human = getattr(messages[i], "content", "")
            ai = getattr(messages[i + 1], "content", "") if i + 1 < len(messages) else ""
            pairs.append(f"Human: {human}\nAssistant: {ai}")
        return "\n\n".join(pairs[-MAX_CONTEXT_TURNS:])

    def _condense_question(self, question: str, history: str) -> str:
        """Rephrase follow-up question as standalone when history exists."""
        if not history:
            return question
        prompt = _CONDENSE_TEMPLATE.format(chat_history=history, question=question)
        result = self._get_llm().invoke(prompt)
        return result.content.strip()

    def chat_stream(self, question: str):
        """
        Stream the answer token by token.
        Yields str chunks. After iteration, self.last_sources holds source docs.
        """
        if self.vectorstore is None:
            yield "No documents loaded yet. Please load or upload documents first."
            return

        history = self._format_history()
        standalone_q = self._condense_question(question, history)

        # Retrieve relevant chunks
        docs = self.vectorstore.similarity_search(standalone_q, k=RETRIEVER_K)
        self.last_sources = docs
        context = "\n\n".join(doc.page_content for doc in docs)

        # Build final prompt
        prompt_text = _QA_TEMPLATE.format(
            context=context,
            chat_history=history,
            question=question,
        )

        # Stream from Groq
        llm = self._get_llm(streaming=True)
        full_answer = ""
        for chunk in llm.stream(prompt_text):
            token = chunk.content
            full_answer += token
            yield token

        # Save to memory after streaming completes
        self.memory.save_context({"input": question}, {"output": full_answer})

    def get_history(self) -> list[tuple[str, str]]:
        messages = self.memory.chat_memory.messages
        pairs = []
        for i in range(0, len(messages) - 1, 2):
            human = getattr(messages[i], "content", "")
            ai = getattr(messages[i + 1], "content", "") if i + 1 < len(messages) else ""
            pairs.append((human, ai))
        return pairs
